# DocLens 1/3: fine-tune LayoutLMv3 on CORD receipts (Colab T4, ~30 min)

Token classification over the receipt's OCR words (text + 2-D position + image patches).
Output: `predictions/layoutlmv3_test.jsonl` and `predictions/gt_test.jsonl`, copied to Drive for notebook 3.

**Runtime → Change runtime type → T4 GPU.**

In [ ]:
REPO_URL = "https://github.com/Harshchoubey55/doclens.git"
QUICK = False   # True: 4 epochs instead of 12

import os
os.chdir("/content")
if not os.path.exists("doclens"):
    !git clone -q {REPO_URL} doclens
os.chdir("/content/doclens")
!git pull -q
!pip install -q -r requirements.txt "datasets>=2.19" "accelerate>=0.30" wandb
!python -m pytest -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/doclens"
os.makedirs(f"{DRIVE}/predictions", exist_ok=True)

# Optional W&B tracking: put WANDB_API_KEY in Colab Secrets (key icon on the left) to enable it
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    REPORT = "wandb"
except Exception:
    REPORT = "none"
print("reporting to:", REPORT)

In [ ]:
import json, time, torch
from datasets import load_dataset
from doclens.cord import DATASET_ID, ID2LABEL, LABEL2ID, LABELS, decode, words_boxes_labels
from doclens.metrics import evaluate
from doclens.predictions import write_jsonl
from doclens.schema import restrict

ds = load_dataset(DATASET_ID)
print(ds)

# ground truth for every method, written once
gt_rows = [{"doc_id": f"test-{i}", "gt": restrict(json.loads(g)["gt_parse"])} for i, g in enumerate(ds["test"]["ground_truth"])]
write_jsonl("predictions/gt_test.jsonl", gt_rows)

# Sanity check: decoding the *gold* BIO labels should reproduce the gold JSON (upper bound of the decoder)
gold = [decode(w, l) for w, _, l in (words_boxes_labels(g) for g in ds["test"]["ground_truth"])]
print("decoder upper bound:", {k: v for k, v in evaluate(gold, [r["gt"] for r in gt_rows]).items() if k != "per_key_f1"})

In [ ]:
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification, Trainer, TrainingArguments

MODEL = "microsoft/layoutlmv3-base"
processor = AutoProcessor.from_pretrained(MODEL, apply_ocr=False)

def encode(batch):
    words, boxes, labels = zip(*(words_boxes_labels(g) for g in batch["ground_truth"]))
    return processor([im.convert("RGB") for im in batch["image"]], list(words), boxes=list(boxes),
                     word_labels=[[LABEL2ID[x] for x in l] for l in labels],
                     truncation=True, padding="max_length", max_length=512)

cols = ds["train"].column_names
train = ds["train"].map(encode, batched=True, batch_size=16, remove_columns=cols)
val = ds["validation"].map(encode, batched=True, batch_size=16, remove_columns=cols)
train.set_format("torch"); val.set_format("torch")
print(train)

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
EPOCHS, BATCH = (4 if QUICK else 12), 4
# warmup as a step count: transformers 5 dropped `warmup_ratio`
warmup = int(0.1 * EPOCHS * -(-len(train) // BATCH))
args = TrainingArguments(
    output_dir="runs/layoutlmv3", per_device_train_batch_size=BATCH, per_device_eval_batch_size=8,
    learning_rate=5e-5, num_train_epochs=EPOCHS, warmup_steps=warmup, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="eval_loss",
    save_total_limit=1, logging_steps=25, report_to=REPORT, run_name="layoutlmv3-cord", dataloader_num_workers=2)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=val)
trainer.train()

## Test-set inference → structured JSON
One label per word (first sub-token), then BIO spans → line items / totals.

In [ ]:
model = trainer.model.eval()
device = model.device
rows = []
for i, ex in enumerate(ds["test"]):
    words, boxes, _ = words_boxes_labels(ex["ground_truth"])
    t0 = time.perf_counter()
    enc = processor(ex["image"].convert("RGB"), words, boxes=boxes, truncation=True, max_length=512, return_tensors="pt")
    word_ids = enc.word_ids(0)
    with torch.no_grad():
        logits = model(**{k: v.to(device) for k, v in enc.items()}).logits[0]
    pred = logits.argmax(-1).tolist()
    labels, seen = ["O"] * len(words), set()
    for tok, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            seen.add(wid)
            labels[wid] = ID2LABEL[pred[tok]]
    rows.append({"doc_id": f"test-{i}", "pred": decode(words, labels), "latency_s": round(time.perf_counter() - t0, 4)})
write_jsonl("predictions/layoutlmv3_test.jsonl", rows)

score = evaluate([r["pred"] for r in rows], [r["gt"] for r in gt_rows])
print(json.dumps(score, indent=1))
if REPORT == "wandb":
    import wandb
    wandb.log({f"test/{k}": v for k, v in score.items() if isinstance(v, float)})
    wandb.finish()

In [ ]:
!cp predictions/*.jsonl {DRIVE}/predictions/
trainer.save_model(f"{DRIVE}/layoutlmv3-cord")
processor.save_pretrained(f"{DRIVE}/layoutlmv3-cord")
!ls {DRIVE}/predictions